# MPC & LQR — Theoretical Foundations

---

## 1. What LQR Actually Is (Again)

The most natural way to frame "best control" is to write down a cost function that penalises both being in a bad state and using too much force, then minimise it:

$$J = \int_0^{\infty} \left( x^T Q x + u^T R u \right) dt$$

Here $Q$ penalises deviations in state (large $\theta$ is expensive, large cart displacement is expensive) and $R$ penalises control effort (large forces are expensive). The ratio $Q/R$ determines the character of the controller — aggressive correction vs gentle nudging. You choose $Q$ and $R$ as design parameters based on what you care about.

The integral running to infinity means: find the control strategy that minimises the total accumulated cost over the entire future, forever. (Reading this point is very important)

This sounds impossibly hard. The mathematical structure that makes it tractable comes from two properties of the problem:

- The system dynamics are **linear** (or linearised around the upright equilibrium)
- The cost function is **quadratic** in state and input

Together (the LQ structure), these force the optimal value function to take a quadratic form $V(x) = x^T P x$. Refer to the video lectures given on the algebraic Riccati equation. You would observe that when we are taking the form x^TPx, the whole integral becomes the sum of two quantities which can both be set to 0. When we set both the quantities are 0, we give the algebraic Riccati equation and the other optimal control law which is mentioned below:

$$PA + A^TP - PBR^{-1}B^TP + Q = 0$$

And the optimal control law comes straight out:

$$u = -\underbrace{R^{-1}B^TP}_{K} x = -Kx$$

This is the **Linear Quadratic Regulator (LQR)**. The sophistication is entirely in the derivation. The resulting controller is maximally simple — a constant matrix multiply, computed once offline, applied forever.

The key conditions that make this work: **linear system + quadratic cost + infinite horizon**. Break any one and the magic collapses. A nonlinear system means the value function is no longer quadratic. A non-quadratic cost means no closed-form minimisation. A finite horizon means $P$ becomes time-varying and $K$ changes with time.

This last point — finite horizon breaking the constant $K$ result — is exactly the direction we are about to go.

---

## 2. Where LQR Falls Short — The Motivation for MPC

LQR is elegant and computationally trivial at runtime. For the linearised inverted pendulum near the upright position, it works well. But it carries assumptions that the real world does not respect:

**The system is not truly linear.** LQR is derived from a linearised model around the upright equilibrium. When the pole is far from vertical — during a large disturbance, or a swing-up manoeuvre — the linearisation becomes inaccurate and LQR's performance degrades.

**There are no constraints in LQR.** A real cart has a finite track — it cannot move beyond some range $\pm x_{\max}$. The motor has a maximum force it can deliver — $|u| \leq F_{\max}$. LQR has no mechanism to respect these limits. It will happily compute a control input that demands more force than the actuator can provide, or a trajectory that drives the cart off the track.

**Clipping is not the answer.** One might think: compute the LQR output, then clip it to the physical limits. But a clipped value is no longer optimal — the solver never considered the region near the constraint boundary when it computed the unconstrained answer. Clipping discards direction information. The result is suboptimal and can destabilise the system.

What we actually need is a controller that:
1. Can use the full nonlinear model, not just a linearisation
2. Natively respects hard limits on force and cart position
3. Still minimises something close to the LQR cost

This is exactly what **Model Predictive Control (MPC)** provides.

---

## 3. The Core Idea of MPC

Instead of solving an infinite-horizon problem once offline, MPC solves a *finite-horizon* problem online at every timestep. At each instant, it looks $N$ steps into the future, finds the sequence of forces $[u_0, u_1, \ldots, u_{N-1}]$ that minimises the cost over those $N$ steps subject to all constraints, applies only the first force $u_0$, then repeats at the next timestep with a fresh measurement.

This is called the **receding horizon** principle — you always plan $N$ steps ahead, but only ever commit to the first step. The horizon moves forward with time.

The feedback is built into the structure: even if the model is slightly wrong, re-solving with fresh state measurements every step compensates for the error. You are never flying blind for more than one timestep.

---

## 4. The Cost Function in MPC

The infinite horizon of LQR is replaced with an explicit $N$-step sum, plus a **terminal cost** that accounts for everything from step $N$ onward:

$$J = \underbrace{\sum_{k=0}^{N-1} \left( x_k^T Q x_k + u_k^T R u_k \right)}_{\text{explicit N steps}} + \underbrace{x_N^T P x_N}_{\text{terminal cost: covers } N \to \infty}$$

The terminal cost $P$ is not a guess — it is the LQR solution to the infinite-horizon problem on the linearised model, computed once offline via the **Discrete Algebraic Riccati Equation (DARE)**. The logic is: by the time the system reaches step $N$, the controller has handled the hard nonlinear transient explicitly. The state should be close enough to the target that the linear approximation is reasonable for the tail. So you borrow LQR's infinite-horizon wisdom to summarise everything from $N$ onward in a single matrix.

U can think of P as the penalty matrix as in LQR. We wil keep it such that the value of the quadratic form x^TPx wil be high when we have a undesirable state at the end of N steps. That is basically to think of as the cost-to-go function. It tells us the amount of "effort" in terms of point which is left for th system to reach the equilibrium state. 

You get the best of both: nonlinear accuracy for the transient (first $N$ steps), and infinite-horizon optimality from LQR for the tail.

---

## 5. Where the Terminal Cost P Comes From — The Value Function

To understand why $P$ has the form it does, consider the **cost-to-go** (also called the value function):

$$V(x) = \text{minimum total cost from state } x \text{ to infinity, under optimal control}$$

Intuition:
- $V(0) = 0$ — already at the target, no future cost
- $V(\text{large angle}) = \text{large number}$ — expensive recovery ahead
- $V(\text{small angle}) = \text{small number}$ — cheap correction needed

**Bellman's key insight** is that you don't need to think about infinity explicitly. The value function satisfies:

$$V(x_k) = \min_u \Big[ \underbrace{x_k^TQx_k + u^TRu}_{\text{cost this step}} + \underbrace{V(x_{k+1})}_{\text{cost from next state onward}} \Big]$$

An optimal plan has the property that from any point along it, the remaining plan is also optimal. The infinite sum telescopes into a one-step problem plus a recursive term.

Guess $V(x) = x^T P x$, substitute into Bellman's equation, differentiate with respect to $u$, set to zero — out comes $u^* = -Kx$, and the condition on $P$ becomes the DARE. $P$ is the unique matrix that is self-consistent under this recursion — a fixed point. It summarises the entire infinite future in a single matrix. Once you have it, you never need to think about infinity again.

---

## 6. Why Discretisation is Structurally Necessary

In continuous time, the control input $u(t)$ is a *function* over an interval — it lives in an infinite-dimensional space. No computer can directly optimise over an infinite-dimensional space.
$$\dot{x} = A_c x + B_c u \xrightarrow{\text{ZOH}} x_{k+1} = A_d x_k + B_d u_k$$

This is where the continuous-time physics gets converted into a discrete timestep model. The continuous system $\dot{x} = Ax + Bu$ has an exact solution involving the matrix exponential $e^{At}$. ZOH assumes the control input $u$ is held constant over each timestep $T$ — which is literally true for a digital controller. Under this assumption, you can evaluate the exact solution over one interval to get

$$A_d = e^{AT}$$
and
$$B_d = A^{-1}(e^{AT} - I)B$$

with no approximation in the dynamics. Euler is just the first-order Taylor expansion of this same matrix exponential
$$e^{AT} \approx I + AT$$
so ZOH is strictly more accurate for the same timestep size.


The matrix exponential `expm(M_aug * dt)` is doing real mathematical work here — integrating the ODE exactly over one interval. This is the approximation step. We are getting the recursive realtion by a very smart assumption about to force vector (we assume force to be constant). Then we will solve a linear ODE with constant coeff. (This will bring in the exp term)

Discretising into $N$ steps replaces the function with a finite vector:

$$\mathbf{U} = [u_0, u_1, u_2, \ldots, u_{N-1}]^T \in \mathbb{R}^N$$

Now the optimisation is over $N$ real numbers. And crucially — physical constraints become simple linear inequalities on this vector:

$$-F_{\max} \leq u_k \leq F_{\max} \quad \forall k = 0, \ldots, N-1$$

$$-x_{\max} \leq x_{\text{cart},k} \leq x_{\max} \quad \forall k = 0, \ldots, N-1$$

This turns the entire problem into a **Quadratic Program (QP)** — minimise a quadratic cost subject to linear inequality constraints. QPs have fast, reliable solvers and are well understood mathematically. Discretisation is not a convenience or an approximation — it is the step that makes the constrained optimisation computationally tractable.

---

## 7. The QP in Concrete Terms

Once you discretise, every future state becomes a linear function of $\mathbf{U}$ alone (given the current state $x_0$):

$$x_1 = Ax_0 + Bu_0$$
$$x_2 = A^2x_0 + ABu_0 + Bu_1$$
$$\vdots$$

Stack all $N$ predicted states into one equation:

$$\mathbf{X} = \mathcal{A}x_0 + \mathcal{B}\mathbf{U}$$

where $\mathcal{A}$ and $\mathcal{B}$ are built by stacking powers of the linearised system matrices. The entire predicted trajectory is now a linear function of $\mathbf{U}$. Substituting into the cost function and expanding:

$$J = \mathbf{U}^T \underbrace{(\mathcal{B}^T\bar{Q}\mathcal{B} + \bar{R})}_{H} \mathbf{U} + 2\underbrace{(\mathcal{B}^T\bar{Q}\mathcal{A}x_0)}_{g}^T \mathbf{U}$$

Two matrices determine everything:
- **H** — the shape of the cost bowl. Computed once offline, never changes.
- **g** — tilts the bowl based on the current state. Recomputed every timestep with one matrix-vector multiply.

The constraints are assembled into a matrix inequality $C\mathbf{U} \leq d$, where $d$ is also partially updated each timestep using $x_0$. The QP handed to the solver at each timestep is:

$$\min_{\mathbf{U}} \quad \frac{1}{2} \mathbf{U}^T H \mathbf{U} + g^T \mathbf{U} \qquad \text{subject to} \quad C\mathbf{U} \leq d$$

This is why MPC is fast enough for real-time: the expensive matrix construction ($H$, $\mathcal{A}$, $\mathcal{B}$) happens offline once at startup. Online, you update $g$ and $d$, call the solver, and extract the first element of $\mathbf{U}^*$.

---

## 8. The Full MPC Loop

```
OFFLINE (once at startup):
  Linearise system → get A, B matrices
  Solve DARE → get P (terminal cost)
  Build prediction matrices 𝒜, ℬ
  Build H = ℬᵀ Q̄ ℬ + R̄   ← fixed forever
  Build constraint matrix C  ← fixed forever

ONLINE (every timestep):
  Read current state x₀
  Compute g = ℬᵀ Q̄ 𝒜 x₀   ← one matrix-vector multiply
  Update d from x₀           ← state constraint right-hand side
  Solve QP: min ½ UᵀHU + gᵀU  subject to  CU ≤ d
  Apply u₀* to the system
  Measure new state → repeat
```

---

## 9. Why the Effective Gain Varies

In LQR, $K$ is constant — `u = -Kx` forever, regardless of where the system is.

In MPC, you never explicitly compute $K$. You solve the QP and get $u_0^*$ directly. The implicit "gain" — the ratio of $u_0^*$ to $x_0$ — varies. The reason is precise:

**The constraints themselves do not change.** $F_{\max}$ stays $F_{\max}$, position limits stay fixed. What changes is **which constraints are active** as the state evolves. Different states push against different constraint boundaries, which changes the geometry of the solution, which changes the effective gain.

- When constraints are inactive: MPC collapses to LQR — the effective gain is the same constant $K$.
- When a constraint is active: the cost bowl is cut by the constraint boundary, the optimum shifts, and the effective gain differs from LQR's $K$.

The mapping from $x_0$ to $u_0^*$ is **piecewise linear** — a different linear map in each region of state space depending on which constraints are binding. MPC implicitly navigates this every timestep without you ever writing down $K$ explicitly.

For the nonlinear case (NMPC), there is a second source of variation: the dynamics themselves change shape as the state moves far from the linearisation point. Both the active constraint set and the local dynamics vary simultaneously — which is why NMPC is strictly more powerful than LQR for a real inverted pendulum.

---

## 10. LQR vs MPC — Summary

| | LQR | Linear MPC | NMPC |
|---|---|---|---|
| Dynamics used | Linearised only | Linearised | Full nonlinear |
| Constraints | Cannot handle | Native | Native |
| Horizon | Infinite (offline) | Finite, receding | Finite, receding |
| Terminal cost | — | DARE P | DARE P (linearised) |
| Online computation | `u = -Kx` only | QP solve | NLP solve |
| Effective gain | Constant $K$ | Constant / piecewise linear | Varies continuously |
